In [1]:
import pandas as pd

In [2]:
df_conversations = pd.read_parquet('conversations.parquet',engine='fastparquet')
df_reactions_full = pd.read_parquet('reactions.parquet', engine='pyarrow')
# df_votes = pd.read_parquet('votes.parquet',engine='fastparquet')

df_merged = pd.merge(df_reactions_full, df_conversations, on='conversation_pair_id', how='left',suffixes=('', '_right'))

In [3]:
# print(df_conversations.columns)
print('\n')
print(df_reactions_full.columns)
print('\n')
# print(df_votes.columns)


print(len(df_reactions_full))
print(len(df_reactions_full[df_reactions_full["creative"]==True]))
print(len(df_reactions_full[df_reactions_full["creative"]==True])/len(df_reactions_full))





Index(['id', 'timestamp', 'model_a_name', 'model_b_name', 'refers_to_model',
       'msg_index', 'opening_msg', 'conversation_a', 'conversation_b',
       'model_pos', 'conv_turns', 'conversation_pair_id', 'conv_a_id',
       'conv_b_id', 'refers_to_conv_id', 'session_hash', 'visitor_id',
       'response_content', 'question_content', 'liked', 'disliked', 'comment',
       'useful', 'creative', 'complete', 'clear_formatting', 'incorrect',
       'superficial', 'instructions_not_followed', 'model_pair_name',
       'msg_rank', 'question_id', 'system_prompt'],
      dtype='str')


94294
6354
0.06738498737989691


In [4]:
df_reactions_full.iloc[60]


id                                                                      176493
timestamp                                           2025-03-25 13:49:04.671683
model_a_name                                     deepseek-r1-distill-llama-70b
model_b_name                                                           qwq-32b
refers_to_model                                  deepseek-r1-distill-llama-70b
msg_index                                                                    1
opening_msg                      bonjour, quel est le meilleur langage de prog
conversation_a               [{'content': 'bonjour, quel est le meilleur la...
conversation_b               [{'content': 'bonjour, quel est le meilleur la...
model_pos                                                                    a
conv_turns                                                                   1
conversation_pair_id         0f139d2e4b16403d8ecae736b60ca3cc-79ef955c54494...
conv_a_id                                     0f139d

In [5]:
df_merged = pd.merge(df_reactions_full, df_conversations, on='conversation_pair_id', how='left',suffixes=('', '_right'))
cols_to_drop = [c for c in df_merged.columns if c.endswith('_right')]
df_merged = df_merged.drop(columns=cols_to_drop)

print(df_merged.columns)

Index(['id', 'timestamp', 'model_a_name', 'model_b_name', 'refers_to_model',
       'msg_index', 'opening_msg', 'conversation_a', 'conversation_b',
       'model_pos', 'conv_turns', 'conversation_pair_id', 'conv_a_id',
       'conv_b_id', 'refers_to_conv_id', 'session_hash', 'visitor_id',
       'response_content', 'question_content', 'liked', 'disliked', 'comment',
       'useful', 'creative', 'complete', 'clear_formatting', 'incorrect',
       'superficial', 'instructions_not_followed', 'model_pair_name',
       'msg_rank', 'question_id', 'system_prompt', 'system_prompt_a',
       'system_prompt_b', 'mode', 'custom_models_selection', 'short_summary',
       'keywords', 'categories', 'languages', 'total_conv_a_output_tokens',
       'total_conv_b_output_tokens', 'model_a_total_params',
       'model_b_total_params', 'model_a_active_params',
       'model_b_active_params', 'total_conv_a_kwh', 'total_conv_b_kwh'],
      dtype='str')


In [6]:
df_true = df_merged[df_merged['creative'] == True]  
df_false = df_merged[df_merged['creative'] == False] 
n_true = len(df_true)//2    
df_false_sample = df_false.sample(n=n_true, random_state=42)
df = pd.concat([df_true.sample(len(df_true)//2), df_false_sample]).sample(frac=1, random_state=42)  # shuffle
print(df['creative'].value_counts())
print(f"Total : {len(df)} lignes")


creative
False    3177
True     3177
Name: count, dtype: int64
Total : 6354 lignes


In [7]:
df.response_content.str.len().mean()

np.float64(2881.290368271955)

# 8. Question 4 — Validité de construit 

Est-ce que le signal creative dans compar:IA mesure vraiment la créativité, ou est il biaisé et capture t il d'autres données (comme la longueur, le formatage, conformisme...)

Pour répondre à cette question, nous procédons en deux étapes :
1. **Validité convergente** : creative doit corréler positivement (r > 0.40) avec des métriques de créativité au sens de la nouveauté lexicale et sémantique (MATTR, N-gram Rarity, distance d'embedding).
2. **Validité discriminante** : creative ne doit pas corréler (r < 0.20) avec des variables sans liens conceptuels (longueur, mise en forme, incorrect).


In [8]:
from collections import Counter
import numpy as np
import spacy
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer
import torch
from torch.nn.functional import cosine_similarity


model = SentenceTransformer("dangvantuan/sentence-camembert-large")
tokenizer = AutoTokenizer.from_pretrained("dangvantuan/sentence-camembert-large")


nlp = spacy.load("fr_core_news_sm")

def tokenize(text):
    doc = nlp(text)
    tokens = [token.text for token in doc]
    return tokens

### Mattr

def moving_average_ttr(tokens,window_size=100):
    if len(tokens)==0:
        return 0
    if len(tokens) < window_size:
        return len(set(tokens)) / len(tokens)
    
    
    ttr_list = []
    n_chunk = len(tokens) // window_size
    
    for i in range(n_chunk):
        window = tokens[i*window_size : (i+1)*window_size]
        ttr_list.append(len(set(window)) / len(window))
    
    if len(tokens) % window_size != 0:
        last_window = tokens[n_chunk*window_size:]
        ttr_list.append(len(set(last_window)) / len(last_window))
    
    return np.mean(ttr_list)

### Implémentation du N-gramme Rarity Score
def corpus_to_Ngram(corpus,N=2):

    Ngram_corpus = [
        " ".join(token_list[i:i+N])
        for token_list in corpus
        for i in range(len(token_list)-N+1)
    ]
    return set(Ngram_corpus)


def N_gram_rarity(token, vocab_corpus, N=2):
    if len(token) == 0:
        return 0
    if len(token) < N:
        Ngram = [" ".join(token)]
    else:
        Ngram = [" ".join(token[i:i+N]) for i in range(len(token)-N+1)]
    
    vocab = set(Ngram)  # ← moved here, always executes
    return len(vocab - vocab_corpus) / len(vocab)


def get_embedding(text, chunk_size=400, overlap=50):
    tokens = tokenizer.encode(text)
    
    if len(tokens) <= 512:
        return torch.tensor(model.encode(text)) 
    
    chunks = []
    for i in range(0, len(tokens), chunk_size - overlap):
        chunk = tokens[i:i + chunk_size]
        chunks.append(tokenizer.decode(chunk, skip_special_tokens=True))
    
    embeddings = model.encode(chunks)
    return torch.tensor(embeddings).mean(dim=0)
def get_corpus_mean_embedding_np(corpus):
    embeddings = [get_embedding(text) for text in corpus]
    embeddings = torch.cat(embeddings, dim=0)
    mean_embedding = embeddings.mean(dim=0)
    return mean_embedding

# Distance d'embedding

def embedding_distance(v1, v2):
    # Convert to tensor if numpy
    if isinstance(v1, np.ndarray):
        v1 = torch.tensor(v1)
    if isinstance(v2, np.ndarray):
        v2 = torch.tensor(v2)
        
    v1 = v1.squeeze().unsqueeze(0)
    v2 = v2.squeeze().unsqueeze(0)
    
    similarity = cosine_similarity(v1, v2)
    return 1.0 - similarity.item()


No sentence-transformers model found with name dangvantuan/sentence-camembert-large. Creating a new one with mean pooling.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

CamembertModel LOAD REPORT from: dangvantuan/sentence-camembert-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
df['tokens'] = df.response_content.apply(lambda x: tokenize(x))

### MATTR

In [10]:
df['MATTR'] = df.tokens.apply(lambda x: moving_average_ttr(x))

### N-gram rarity

In [11]:
from datasets import load_dataset

ds = load_dataset("wikipedia", "20220301.fr", split="train", streaming=True)


N = 500
ds_shuffled = ds.shuffle(seed=42, buffer_size=10000)
dataset_fr = list(ds_shuffled.take(N))


c:\Users\youce\miniconda3\Lib\site-packages\datasets\load.py:1491: FutureWarning: The repository for wikipedia contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/wikipedia
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


In [12]:
from collections import Counter

def build_ngram_vocab_fast(dataset, nlp, N=2, limit=50000, batch_size=1000):
    counter = Counter()
    texts = []

    for i, x in enumerate(dataset):
        texts.append(x["text"])

        if len(texts) == batch_size:
            for doc in nlp.pipe(texts):
                tokens = [t.text.lower() for t in doc if t.is_alpha]
                ngrams = [" ".join(tokens[j:j+N]) for j in range(len(tokens)-N+1)]
                counter.update(ngrams)
            texts = []

        if i >= limit:
            break

    # Traiter le reste
    if texts:
        for doc in nlp.pipe(texts):
            tokens = [t.text.lower() for t in doc if t.is_alpha]
            ngrams = [" ".join(tokens[j:j+N]) for j in range(len(tokens)-N+1)]
            counter.update(ngrams)

    return set(counter)



print(f"Nombre d'exemples FR (train) : {len(dataset_fr)}")

# Construire les n-grammes
ngram_freq = build_ngram_vocab_fast(dataset_fr, nlp, N=2, limit=50000)

# print(f"\nTop 20 bigrammes :")
# for ngram, count in ngram_freq.most_common(20):
#     print(f"  {ngram}: {count}")

Nombre d'exemples FR (train) : 500


In [13]:
df['N_gram_rarity'] = df.tokens.apply(lambda x: N_gram_rarity(x,ngram_freq))

### Embedding distance

In [14]:
response = df.iloc[0].response_content
question = df.iloc[0].question_content
# print(get_embedding(response))

In [15]:
def embedding_row(row):
    response = get_embedding(row.response_content)
    # print(response.shape)
    question = get_embedding(row.question_content)
    # print(question.shape)
    return embedding_distance(response,question)

df['embedding_distance']= df.apply(embedding_row,axis=1)

In [16]:
df.N_gram_rarity

24335    0.757050
76660    0.823529
92930    0.821549
56482    0.947802
44201    1.000000
           ...   
3192     0.673077
13894    0.849010
42991    0.806452
10370    0.860656
8535     0.788644
Name: N_gram_rarity, Length: 6354, dtype: float64

In [17]:
corr_mattr = df.creative.corr(df.MATTR)
corr_ngram = df.creative.corr(df.N_gram_rarity)
corr_embedding = df.creative.corr(df.embedding_distance)


print(corr_mattr)
print(corr_ngram)
print(corr_embedding)

0.01608761063853142
0.10230483253937915
0.04648037805448873


### 8.2 Tests de validité discriminante

In [18]:
df.columns

Index(['id', 'timestamp', 'model_a_name', 'model_b_name', 'refers_to_model',
       'msg_index', 'opening_msg', 'conversation_a', 'conversation_b',
       'model_pos', 'conv_turns', 'conversation_pair_id', 'conv_a_id',
       'conv_b_id', 'refers_to_conv_id', 'session_hash', 'visitor_id',
       'response_content', 'question_content', 'liked', 'disliked', 'comment',
       'useful', 'creative', 'complete', 'clear_formatting', 'incorrect',
       'superficial', 'instructions_not_followed', 'model_pair_name',
       'msg_rank', 'question_id', 'system_prompt', 'system_prompt_a',
       'system_prompt_b', 'mode', 'custom_models_selection', 'short_summary',
       'keywords', 'categories', 'languages', 'total_conv_a_output_tokens',
       'total_conv_b_output_tokens', 'model_a_total_params',
       'model_b_total_params', 'model_a_active_params',
       'model_b_active_params', 'total_conv_a_kwh', 'total_conv_b_kwh',
       'tokens', 'MATTR', 'N_gram_rarity', 'embedding_distance'],
      dt

In [19]:
corr_clear_formatting = df.creative.corr(df.clear_formatting)
corr_incorrect = df.creative.corr(df.incorrect)
corr_length_a = df.creative.corr(df.total_conv_b_output_tokens)
corr_length_b = df.creative.corr(df.total_conv_a_output_tokens)

print(corr_clear_formatting)
print(corr_incorrect)
# print(corr_length_a)
print(corr_length_b)



0.2210214268576878
-0.257808566183483
0.1501207304635858


### Analyse des résultats :

### Tableau récapitulatif : Validité convergente

| Métrique | Corrélation avec `creative` | Seuil attendu | Résultat |
|---|---|---|---|
| MATTR | **-0.016** | r > 0.40 | Non atteint |
| N-gram Rarity Score | **+0.10** | r > 0.40 | Non atteint |
| Distance d'embedding | **+0.04** | r > 0.40 | Non atteint |


**Analyse :**

Aucune des trois métriques ne corèle avec la créativité évaluée par les utilisateurs. La validité convergente n'est pas établie sur ce sous-échantillon de 2000 observations. Le label ne mesure donc pas la créativité au sens de la diversité sémantique et lexicale.

### Tableau récapitulatif : Validité discriminante

| Variable | Corrélation avec `creative` | Seuil attendu | Résultat |
|---|---|---|---|
| formattage clair | **+0.22** | r < 0.20 | non-atteint |
| incorrect | **-0.26** | r < 0.20 | non-atteint |
| longueur | **+0.15** | r < 0.20 | Atteint |

**Analyse :**

On a des valeurs pas mal correlées nous indiquant que les utilisateurs semblent tomber dans les biais faisant croire à de la créativité

## Exercice 4.3 — Le paradoxe du juge dans le contexte compar:IA

### Le paradoxe du juge


Le paradoxe du juge (Boden 1994) stipule qu'**un évaluateur ne peut reconnaître une réponse vraiment nouvelle que s'il dispose déjà des structures cognitives adéquates pour l'apprécier**. Autrement dit : reconnaître l'originalité nécessite une référence. Sans cela, ce qui est nouveau passe inaperçu ou pire, est perçu comme bizarre ou incorrect.

### Forces

- Les labels correspondent à des jugements humains réels
- La diversification et le grand nombre d'anotateurs permet de réduire certains des possibles biais. En particulier des anotateurs non-experts car les experts auront potentiellement tendance à privilégier des réponses conformes aux normes de leur discipline (biais possible avec la variable incorrect qui est évité)

### Faiblesses

- Si les réponses sont rarement interprétées comme créatives par les utilisateurs, le signal est difficile a interpréter
- Un utilisateur non-expert peut justement ne pas avoir la référence nécessaire à l'évaluation de la créativité d'une réponse
- Les standards de plusieurs utilisateurs peuvent être très différents
